# C10 測試日（test02）：競賽二（棋力預測）逐盤輸出（Colab 版，只推論）

**用途**：`docs/PREREG_02_OPENING_FUSION.md` 第 9 節（rev 5）的 Colab 部分。用 prereg02 指定的權重 `epoch_9.pth`（不訓練、不調參），算三組題目的逐盤輸出（8 種對稱的 logits）：

1. `seam_val`：本機驗證題 1,589 題（接縫核對用，和 prereg02 當時算的比）。
2. `public`、`private`：官方測試題各 400 題。

合併、校準、開局特徵、規則 B 和交件檔都在本機做（`tools/t02_rank.py`），不在這裡。這份 notebook 不讀訓練資料，也不讀任何答案檔。

## 怎麼跑

0. 用**跑 prereg02 的同一個 Google 帳號**：權重和接縫參照在它的 Drive（`MyDrive/C10/results/prereg02/rank/`）。
1. `MyDrive/C10/` 裡要有：`C10_bundle_v1.zip`（原本那份，不用重傳）、Claude 給你的 `C10_test_v1.zip`（檔名不能改、不要解壓），和這份 `test02_rank.ipynb`。
2. 選單「執行階段 → 變更執行階段類型」：選 **G4 GPU**。G4 是灰色、點不了時，選 **A100 GPU**，「大量 RAM」關掉。沒有 GPU 或 Colab 改派別的 GPU 時，前面幾格就會停下。
3. 選單「執行階段 → 全部執行」。第一個程式格會要求存取 Google Drive，按允許。
4. 跑完後（G4 約 10 分鐘，推估），最後兩行是「Drive 已寫入完成並卸載」和「完成。請下載…」。先按「執行階段 → 中斷連線並刪除執行階段」，再下載 `MyDrive/C10/results/test02/test02_rank_results.zip`，把位置告訴 Claude。
5. 跑完前不要關分頁，電腦不要睡眠。

詳細步驟以 `docs/COLAB_RUN.md` 的「測試集（test02）」一節為準；和這裡不一致時，以那一節為準。

## 會停下來的情況（錯誤訊息裡都寫了處理方式）

依序：
- **找不到測試集 zip，或 zip 的種類不對**：在第一個程式格停下。
- **沒有 GPU，或 GPU 不是 G4／A100**：還沒開始算。先刪除執行階段，換成 G4（灰色時 A100）再全部執行。
- **權重或接縫參照的 SHA-256 不符、測試集 zip 的檔案和 MANIFEST 不符、題目檔格式不對、Drive 空間不足、只讀目錄和第一次執行時不同**：preflight 格停下，還沒開始算。
- **上一次執行的接縫核對沒過**（`seam_check.json` 記沒過）：停下。舊的逐盤輸出不能沿用，重跑必須重算：照 Claude 的指示先把 Drive 的 `C10/results/test02/rank` 資料夾改名成 `rank_failed_1`，再全部執行。
- **某一組已有逐盤輸出，但題目檔換過**（`_INPUT.json` 不同）：停下，不會沿用舊的輸出。
- **任何寫進 prereg01、prereg02 目錄的動作**：一律擋下。
- **接縫核對沒過**（包括接續後三組的逐盤輸出不是同一個 GPU 型號、torch、cuDNN 算的）：這次的測試輸出一律不用、不上傳。會先打包 zip 再停下。**不要自己重跑**，下載 zip、截圖給 Claude。
- **結尾發現 prereg01 或 prereg02 目錄被改動**：結果照樣打包，但會停下；截圖給 Claude。

出現紅字時都先截圖給 Claude，不要自己重跑。

## 結果寫到哪裡

`MyDrive/C10/results/test02/rank/`（彩排是 `test02_rehearsal/`）：

| 檔案 | 內容 |
|---|---|
| `<組>/dumps/val_keys.npz` | 每個 slot 一列：`question_id`、`slot`（從 1 起算）、`color`、`parsed`；題目檔的列順序，再依 slot 順序 |
| `<組>/dumps/val_p02e9_d4.npz` | `logits` float32 [盤數, 8, 10]、`weights_sha256`、`syms`；解析失敗的 slot 整列是 NaN |
| `<組>/dumps/_DONE_<檔名>.json` | 完成標記：列數、SHA-256、權重 SHA-256、秒數，以及算它的那次執行的 GPU 型號、torch、cuDNN |
| `<組>/dumps/_INPUT.json` | 這一組用的題目檔和它的 SHA-256 |
| `seam_check.json` | 接縫核對的數字和結果 |
| `readonly_before.json`、`readonly_after.json` | prereg01、prereg02 目錄每個檔的大小和 mtime（開頭、結尾） |
| `config.json`、`env.json`、`run_log.json`、`timings.json` | 設定、環境、每次執行的紀錄（`status`）、各階段秒數 |

`<組>` 是 `seam_val`、`public`、`private`。`MyDrive/C10/results/test02/test02_rank_results.zip`：上面全部檔案（zip 內路徑從 `rank/` 起），這個檔帶回本機。

## 中斷後怎麼接續

重新打開後，**選和中斷前同一種 GPU**（中斷前是 G4 就選 G4），再按一次「全部執行」：已完成的逐盤輸出靠完成標記、SHA-256 和 `_INPUT.json` 跳過；沒完成的重算。三組的逐盤輸出必須是同一個執行環境算的（GPU 型號、torch、cuDNN，記在各組的完成標記）：接續時換了 GPU（例如中斷前 G4、接續時 A100），或 Colab 換了 torch／cuDNN，接縫核對會判定沒過並停下，這時照 Claude 的指示先把 `C10/results/test02/rank` 改名成 `rank_failed_1`，再整份重算。中斷前那一種 GPU 拿不到時，先不要跑，告訴 Claude。

## 和 prereg02 notebook 的差異

底稿是已 commit 的 `colab/prereg02_rank.ipynb`（產生時核對 SHA-256）。沿用的格子沒列在下表的，和 prereg02 逐字相同；和官方 notebook、prereg01 notebook 的差異照 prereg02 的差異表。逐格的 unified diff 在 `results/test02/notebook_diff_rank.txt`。

| 格 | 改了什麼 |
|---|---|
| 新增 | 第一格 markdown：用途、步驟、GPU、會停下的情況、輸出清單、交回方式、這張差異表。 |
| 設定 | 先讀 Drive 上 `C10_test_v1.zip` 的 `tests/MANIFEST.json` 決定種類（正式 `official`、彩排 `rehearsal`；SMOKE 只接受 `smoke`），找不到或種類不對就停。結果目錄改成 `test02`（彩排 `test02_rehearsal`，SMOKE `test02_smoke`）的 `rank/`，沒有 `ckpt/`、沒有分片備份。只讀目錄改成兩個：`prereg01`、`prereg02`（SMOKE 是 `_smoke`），`guard_write()` 兩個都擋，`readonly_fingerprint(root)` 取代 `p01_fingerprint()`。`sha256_path()`、`gpu_name_of()` 從 prereg02 的 preflight 格逐字搬到這裡（本版的 preflight 在 GPU 關卡之後）。拿掉收尾開關 `FINISH_WITHOUT_TRAINING` 和 `SMOKE_CFG_P02`；`P02_CHANGES` 換成 `T02_CHANGES`，另記這份 notebook 的 SHA-256（規則見 `NOTEBOOK_SHA256_RULE`）。`config.json` 的 `prereg` 改指第 9 節，記 `rehearsal`、測試集種類、底稿 SHA-256、這張表和只讀目錄。`run_log.json` 頂層加 `status`（這次執行開始時清空，結束時寫 completed／seam_failed／readonly_changed）。 |
| 準備 | 拿掉解開訓練資料的整段（本版只推論，不讀訓練資料）；第一行註解跟著改。打包檔的 SHA-256 核對和拒收答案檔的那一行照舊。 |
| 底稿官方 2 | 拿掉 `CKPT_DIR`、`guard_write(CKPT_DIR)`／`makedirs`、`EXTRA_EPOCHS`／`PEAK_LR`／`FINAL_EPOCH` 和它們的 assert。沒有 GPU 的關卡拿掉收尾開關的例外；型號關卡照舊（只接受 G4〔RTX PRO 6000〕或 A100）。最後的 `print` 只印裝置、GPU、NPROC。 |
| 新增 preflight | 在 GPU 關卡之後：① 權重 `prereg02/rank/ckpt/epoch_9.pth` 的 SHA-256 等於登記的值（SMOKE：等於 prereg02_smoke 完成標記記的）；② 接縫參照 `prereg02/rank/dumps/val_keys.npz`、`val_p02e9_d4.npz` 的 SHA-256 釘死，prereg02 的 GPU 型號記下；③ 測試集 zip 解到 `WORK/tests/`，只收 `tests/` 底下的 6 個官方檔名和 `MANIFEST.json`，每個檔的 SHA-256、大小要等於 MANIFEST；④ 快速格式檢查（T1 的一部分：沒有 BOM、400 題、`question_id` 不重複且前綴是 `pub_`／`priv_`、`num_games` 是 5–20 的整數、`sgf_1..20`／`color_1..20` 欄都在、剛好前 `num_games` 格有棋、執色是 B/W、兩份題目不重疊）；⑤ 上一次執行的 `seam_check.json` 沒過就停（已完成的 dump 會被沿用，登記的「重跑一次」必須重算：先把結果資料夾改名保留）；每組的 `_INPUT.json`（題目檔 SHA-256）：已存在但不同就停（防止沿用換過題目檔的舊 dump）；⑥ Drive 剩餘空間 ≥ 0.5 GB；⑦ 兩個只讀目錄的開頭指紋 `readonly_before.json`，重跑時和第一次記的不同就停。`config.json` 頂層記 GPU、torch、cuDNN、權重、測試集 zip 與 MANIFEST 的 SHA-256、各組題目檔 SHA-256 與題數。 |
| 底稿 dump 工具 | 拿掉 inner 的部分（`EXPECTED_INNER_GAMES`、`INNER_*`、`decode_with_key`、`KEY_LOADER`、`dump_inner`）和最後的 print。`DUMP_DIR` 不在這格建立，改由各組的格子在呼叫前指定（底稿的函式把它當全域變數讀，本身一字不改）。`parse_val()` 改成 `parse_questions(path)`（依題目檔路徑快取，`pd.read_csv(path, dtype={'question_id': str})`，SMOKE 照舊只取前 12 題）；`dump_val(net, …)` 改成 `dump_questions(net, path, …)`，檔名仍是 `val_` 開頭（在 test02 裡只代表「一組題目的逐盤輸出」）。`record_p01_after()` 改成兩個只讀目錄的 `record_readonly_after()`。`d4`、`load_net`、`sgf_to_features`、`dump_is_done`、`write_dump`、`write_keys`、`pack_results`、`flush_drive`、解析迴圈和逐題推論迴圈和底稿逐字相同（產生器用 ast 核對）。 |
| 新增三組逐盤輸出 | `seam_val`（打包檔的本機驗證題 1,589 題）、`public`、`private` 各一格，內容相同、只有第一行不同：`DUMP_DIR = <組>/dumps`，`write_keys('val_keys', …)`，`dump_questions(…, 'p02e9_d4', 8, …)`；權重只載入一次（`load_net(path, expect_sha=登記的 SHA-256)`）。 |
| 新增接縫核對 | `seam_val` 的逐盤輸出和 Drive 上 prereg02 的比（登記第 9 節第 2 條）：key 陣列完全相同；原方向逐盤 softmax 的每題 float32 平均最大差 ≤ 1e-4（算這份 dump 的 GPU 型號和 prereg02 相同時）或 ≤ 1e-3（不同時）；規則 B 不同 ≤ 4 題（SMOKE 依題數等比例，至少 1）。另記是否逐位元組相同、8 種對稱的 logits 最大差、兩邊的 GPU／torch／cuDNN。三組 dump 的完成標記記的 GPU 型號、torch、cuDNN 必須和 `seam_val` 的相同（中斷後接續時，已完成的組會被沿用，可能是別的執行環境算的；不同就算接縫沒過，接縫才涵蓋產生測試輸出的環境）。寫 `seam_check.json`；沒過就記結尾指紋、`status = seam_failed`、打包、送出 Drive 的寫入後停下。 |
| 底稿結尾（改寫） | 核對三組 dump 的完成標記、SHA-256、列數、形狀、權重、NaN 位置和 `_INPUT.json`；記 `readonly_after.json` 並和開頭比對（不同就 `status = readonly_changed`，打包後停下）；補完 config／timings／run_log；打包 `test02/test02_rank_results.zip`（zip 內路徑從 `rank/` 起；跳過 `.pth`、`.pt`、`.tmp` 和 ≥ 100 MB 的檔），確認必要的檔都在 zip 裡；最後 `drive.flush_and_unmount()`。 |
| 拿掉 | prereg02 的 preflight、官方 4、6、8、9、11、12、14、15（讀訓練資料、抽分片、loader、建模型、驗證）、分片備份工具、P1 輸出與重現核對、接著訓練、P2 輸出。本版不訓練、不讀訓練資料。 |
| metadata | notebook metadata 沿用 prereg02。每格 cell id 是 `c10-rank-t02-NN`，cell metadata `c10` 記底稿的格號、底稿的 `c10` 和是否改動。 |

## 其他

- **本機冒煙測試**：`colab/run_local_smoke.py` 設定 `C10_SMOKE=1`，只用極少量資料在 CPU 上跑：要先有 `prereg01_smoke`、`prereg02_smoke` 和種類 `smoke` 的測試集 zip；每組只推論前 12 題，結果寫到 `results/test02_smoke/`。
- **不在這裡打分**：答案檔不上傳 Colab；測試集沒有答案。

In [ ]:
# 設定：判斷在 Colab 還是本機冒煙測試，決定路徑和結果目錄。Colab 和本機只有這一格的寫法不同。
import csv
import datetime
import json
import multiprocessing
import os
import shutil
import sys
import time

RUN_START = time.time()


def now_iso():
    return datetime.datetime.now().astimezone().isoformat(timespec='seconds')


IN_COLAB = 'google.colab' in sys.modules
SMOKE = os.environ.get('C10_SMOKE') == '1'   # Colab 上沒有這個環境變數，所以永遠是 False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/C10'
    WORK = '/content/c10'
else:
    DRIVE_DIR = os.environ['C10_DRIVE_DIR']
    WORK = os.environ['C10_WORK_DIR']
    if os.name == 'nt' and not SMOKE:
        # 官方的多進程依賴 Linux 的 fork；原生 Windows 只能跑冒煙測試（測試器會把進程換成執行緒）
        raise RuntimeError('原生 Windows 不支援正式執行，請在 Colab 上跑')

# 官方的多進程（抽特徵的 Process、推論的 Pool、DataLoader 的 worker）用的是 notebook 內定義的函式，只有 fork 能傳給子進程。
# Python 3.14 起 Linux 預設改成 forkserver，所以明確指定 fork；3.13 以前本來就是 fork，行為不變。
if sys.platform.startswith('linux'):
    multiprocessing.set_start_method('fork', force=True)
MP_START_METHOD = multiprocessing.get_start_method()

# 官方 notebook 用相對路徑（./training、./rank-features），這裡一律在 WORK 底下執行。
# webdataset 會把 Windows 路徑的「C:」當成網址格式，所以本機冒煙測試也必須用相對路徑。
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

# 本版（test02）：測試集 zip（Drive 上的 C10_test_v1.zip）。這裡只讀它的 tests/MANIFEST.json，決定這次是正式、彩排還是冒煙；
# 逐檔核對在 preflight 格
import zipfile

TEST_ZIP = os.path.join(DRIVE_DIR, 'C10_test_v1.zip')
TEST_MANIFEST_MEMBER = 'tests/MANIFEST.json'
TEST_MANIFEST_FORMAT = 'c10-test02 manifest v1'
OFFICIAL_TEST_NAMES = ["rank_prediction_test_public.csv", "rank_prediction_test_private.csv", "player_identification_test_public.csv", "player_identification_test_private.csv", "player_identification_candidates_public.csv", "player_identification_candidates_private.csv"]
TEST_FILES = {'public': 'rank_prediction_test_public.csv', 'private': 'rank_prediction_test_private.csv'}
TEST_HELP = ('處理方式：確認 MyDrive/C10/ 裡有 Claude 給你的 C10_test_v1.zip（檔名不能改、不要解壓），再按一次「全部執行」。'
             '還是不對就把這段訊息截圖給 Claude。')
if not os.path.isfile(TEST_ZIP):
    raise SystemExit('找不到測試集 zip：%s。%s' % (TEST_ZIP, TEST_HELP))
try:
    with zipfile.ZipFile(TEST_ZIP) as _z:
        TEST_MANIFEST = json.loads(_z.read(TEST_MANIFEST_MEMBER))
except (OSError, KeyError, ValueError, zipfile.BadZipFile) as e:
    raise SystemExit('測試集 zip 讀不出 %s（%s: %s）。%s' % (TEST_MANIFEST_MEMBER, type(e).__name__, e, TEST_HELP))
TEST_KIND = TEST_MANIFEST.get('kind') if isinstance(TEST_MANIFEST, dict) else None
_KINDS_OK = ('smoke',) if SMOKE else ('official', 'rehearsal')
if not isinstance(TEST_MANIFEST, dict) or TEST_MANIFEST.get('format') != TEST_MANIFEST_FORMAT or TEST_KIND not in _KINDS_OK:
    raise SystemExit('測試集 zip 的 MANIFEST 不是這次能用的（格式 %r、種類 %r；這次只接受 %s）。%s'
                     % (TEST_MANIFEST.get('format') if isinstance(TEST_MANIFEST, dict) else None, TEST_KIND,
                        '、'.join(_KINDS_OK), TEST_HELP))
REHEARSAL = TEST_KIND == 'rehearsal'

# 結果目錄：正式、彩排、SMOKE 分開，絕不混用。本版（test02）只寫 test02（彩排 test02_rehearsal、SMOKE test02_smoke）
RESULT_ROOT = os.path.join(DRIVE_DIR, 'results',
                           'test02_smoke' if SMOKE else ('test02_rehearsal' if REHEARSAL else 'test02'))
RESULT_DIR = os.path.join(RESULT_ROOT, 'rank')

# 本版：prereg01、prereg02 的結果目錄只讀（prereg02 的 epoch_9.pth 是這次的權重，它的驗證題逐盤輸出是接縫參照）
P01_ROOT = os.path.join(DRIVE_DIR, 'results', 'prereg01_smoke' if SMOKE else 'prereg01')
P02_ROOT = os.path.join(DRIVE_DIR, 'results', 'prereg02_smoke' if SMOKE else 'prereg02')
READONLY_ROOTS = {'prereg01': P01_ROOT, 'prereg02': P02_ROOT}


def _real(path):
    return os.path.normcase(os.path.realpath(path))


def _is_under(path, root):
    p, r = _real(path), _real(root)
    return p == r or p.startswith(r.rstrip(os.sep) + os.sep)


def guard_write(path):
    # 本版：寫檔、np.savez、zip、rmtree 的目的地一律先過這一關；落在 prereg01 或 prereg02 底下就停
    for _name, _root in READONLY_ROOTS.items():
        if _is_under(path, _root):
            raise PermissionError('本版不能寫入 %s 的目錄（只讀）：%s' % (_name, path))
    return path


def readonly_fingerprint(root):
    # 只讀目錄每個檔的 {相對路徑: [bytes, mtime_ns]}；開頭和結尾各記一次，結尾必須相同
    out = {}
    for base, dirs, files in os.walk(root):
        dirs.sort()
        for n in sorted(files):
            p = os.path.join(base, n)
            st = os.stat(p)
            out[os.path.relpath(p, root).replace(os.sep, '/')] = [st.st_size, st.st_mtime_ns]
    return out


# 本版：這兩個函式原本在 prereg02 的 preflight 格（本版的 preflight 在 GPU 關卡之後，官方 2 就要用 gpu_name_of）
def sha256_path(path):
    d = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            d.update(chunk)
    return d.hexdigest()


def gpu_name_of(env_run):
    # env.json 一筆執行紀錄裡的 GPU 型號（nvidia-smi 的第一欄）；沒有 GPU 時是 None
    smi = (env_run.get('gpu') or {}).get('nvidia_smi')
    return smi[0].split(',')[0].strip() if smi else None


# 本機工作目錄、test02 的結果目錄都不能和只讀目錄互相包含
for _p in (WORK, RESULT_ROOT):
    guard_write(_p)
    for _root in READONLY_ROOTS.values():
        assert not _is_under(_root, _p), (_root, _p)
guard_write(RESULT_DIR)
os.makedirs(RESULT_DIR, exist_ok=True)

# SMOKE 的縮小方式：只在本機冒煙測試生效，正式執行時完全不用
SMOKE_CFG = {
    'fit_games': 2000,     # load_rows 全部通過之後，才用固定 seed 抽
    'inner_games': 400,
    'sample_seed': 42,
    'nproc': 2,            # 抽特徵和推論的進程數
    'per_shard': 300,      # 讓每個 worker 寫出好幾個分片，測到換檔
    'batch_size': 32,
    'num_epochs': 1,
    'questions': 12,       # 推論只做前 12 題
    'loader_workers': 0,   # 本機 Windows 的 spawn 無法 pickle notebook 內定義的函式
}

# 本版（test02）和 prereg02 notebook 的差異（與第一格的表同一份）
T02_CHANGES = [
    "[新增] 第一格 markdown：用途、步驟、GPU、會停下的情況、輸出清單、交回方式、這張差異表。",
    "[設定] 先讀 Drive 上 `C10_test_v1.zip` 的 `tests/MANIFEST.json` 決定種類（正式 `official`、彩排 `rehearsal`；SMOKE 只接受 `smoke`），找不到或種類不對就停。結果目錄改成 `test02`（彩排 `test02_rehearsal`，SMOKE `test02_smoke`）的 `rank/`，沒有 `ckpt/`、沒有分片備份。只讀目錄改成兩個：`prereg01`、`prereg02`（SMOKE 是 `_smoke`），`guard_write()` 兩個都擋，`readonly_fingerprint(root)` 取代 `p01_fingerprint()`。`sha256_path()`、`gpu_name_of()` 從 prereg02 的 preflight 格逐字搬到這裡（本版的 preflight 在 GPU 關卡之後）。拿掉收尾開關 `FINISH_WITHOUT_TRAINING` 和 `SMOKE_CFG_P02`；`P02_CHANGES` 換成 `T02_CHANGES`，另記這份 notebook 的 SHA-256（規則見 `NOTEBOOK_SHA256_RULE`）。`config.json` 的 `prereg` 改指第 9 節，記 `rehearsal`、測試集種類、底稿 SHA-256、這張表和只讀目錄。`run_log.json` 頂層加 `status`（這次執行開始時清空，結束時寫 completed／seam_failed／readonly_changed）。",
    "[準備] 拿掉解開訓練資料的整段（本版只推論，不讀訓練資料）；第一行註解跟著改。打包檔的 SHA-256 核對和拒收答案檔的那一行照舊。",
    "[底稿官方 2] 拿掉 `CKPT_DIR`、`guard_write(CKPT_DIR)`／`makedirs`、`EXTRA_EPOCHS`／`PEAK_LR`／`FINAL_EPOCH` 和它們的 assert。沒有 GPU 的關卡拿掉收尾開關的例外；型號關卡照舊（只接受 G4〔RTX PRO 6000〕或 A100）。最後的 `print` 只印裝置、GPU、NPROC。",
    "[新增 preflight] 在 GPU 關卡之後：① 權重 `prereg02/rank/ckpt/epoch_9.pth` 的 SHA-256 等於登記的值（SMOKE：等於 prereg02_smoke 完成標記記的）；② 接縫參照 `prereg02/rank/dumps/val_keys.npz`、`val_p02e9_d4.npz` 的 SHA-256 釘死，prereg02 的 GPU 型號記下；③ 測試集 zip 解到 `WORK/tests/`，只收 `tests/` 底下的 6 個官方檔名和 `MANIFEST.json`，每個檔的 SHA-256、大小要等於 MANIFEST；④ 快速格式檢查（T1 的一部分：沒有 BOM、400 題、`question_id` 不重複且前綴是 `pub_`／`priv_`、`num_games` 是 5–20 的整數、`sgf_1..20`／`color_1..20` 欄都在、剛好前 `num_games` 格有棋、執色是 B/W、兩份題目不重疊）；⑤ 上一次執行的 `seam_check.json` 沒過就停（已完成的 dump 會被沿用，登記的「重跑一次」必須重算：先把結果資料夾改名保留）；每組的 `_INPUT.json`（題目檔 SHA-256）：已存在但不同就停（防止沿用換過題目檔的舊 dump）；⑥ Drive 剩餘空間 ≥ 0.5 GB；⑦ 兩個只讀目錄的開頭指紋 `readonly_before.json`，重跑時和第一次記的不同就停。`config.json` 頂層記 GPU、torch、cuDNN、權重、測試集 zip 與 MANIFEST 的 SHA-256、各組題目檔 SHA-256 與題數。",
    "[底稿 dump 工具] 拿掉 inner 的部分（`EXPECTED_INNER_GAMES`、`INNER_*`、`decode_with_key`、`KEY_LOADER`、`dump_inner`）和最後的 print。`DUMP_DIR` 不在這格建立，改由各組的格子在呼叫前指定（底稿的函式把它當全域變數讀，本身一字不改）。`parse_val()` 改成 `parse_questions(path)`（依題目檔路徑快取，`pd.read_csv(path, dtype={'question_id': str})`，SMOKE 照舊只取前 12 題）；`dump_val(net, …)` 改成 `dump_questions(net, path, …)`，檔名仍是 `val_` 開頭（在 test02 裡只代表「一組題目的逐盤輸出」）。`record_p01_after()` 改成兩個只讀目錄的 `record_readonly_after()`。`d4`、`load_net`、`sgf_to_features`、`dump_is_done`、`write_dump`、`write_keys`、`pack_results`、`flush_drive`、解析迴圈和逐題推論迴圈和底稿逐字相同（產生器用 ast 核對）。",
    "[新增三組逐盤輸出] `seam_val`（打包檔的本機驗證題 1,589 題）、`public`、`private` 各一格，內容相同、只有第一行不同：`DUMP_DIR = <組>/dumps`，`write_keys('val_keys', …)`，`dump_questions(…, 'p02e9_d4', 8, …)`；權重只載入一次（`load_net(path, expect_sha=登記的 SHA-256)`）。",
    "[新增接縫核對] `seam_val` 的逐盤輸出和 Drive 上 prereg02 的比（登記第 9 節第 2 條）：key 陣列完全相同；原方向逐盤 softmax 的每題 float32 平均最大差 ≤ 1e-4（算這份 dump 的 GPU 型號和 prereg02 相同時）或 ≤ 1e-3（不同時）；規則 B 不同 ≤ 4 題（SMOKE 依題數等比例，至少 1）。另記是否逐位元組相同、8 種對稱的 logits 最大差、兩邊的 GPU／torch／cuDNN。三組 dump 的完成標記記的 GPU 型號、torch、cuDNN 必須和 `seam_val` 的相同（中斷後接續時，已完成的組會被沿用，可能是別的執行環境算的；不同就算接縫沒過，接縫才涵蓋產生測試輸出的環境）。寫 `seam_check.json`；沒過就記結尾指紋、`status = seam_failed`、打包、送出 Drive 的寫入後停下。",
    "[底稿結尾（改寫）] 核對三組 dump 的完成標記、SHA-256、列數、形狀、權重、NaN 位置和 `_INPUT.json`；記 `readonly_after.json` 並和開頭比對（不同就 `status = readonly_changed`，打包後停下）；補完 config／timings／run_log；打包 `test02/test02_rank_results.zip`（zip 內路徑從 `rank/` 起；跳過 `.pth`、`.pt`、`.tmp` 和 ≥ 100 MB 的檔），確認必要的檔都在 zip 裡；最後 `drive.flush_and_unmount()`。",
    "[拿掉] prereg02 的 preflight、官方 4、6、8、9、11、12、14、15（讀訓練資料、抽分片、loader、建模型、驗證）、分片備份工具、P1 輸出與重現核對、接著訓練、P2 輸出。本版不訓練、不讀訓練資料。",
    "[metadata] notebook metadata 沿用 prereg02。每格 cell id 是 `c10-rank-t02-NN`，cell metadata `c10` 記底稿的格號、底稿的 `c10` 和是否改動。"
]

# 本版：這份 notebook 的 SHA-256（產生器寫入；規則見 NOTEBOOK_SHA256_RULE）
NOTEBOOK_REL = 'colab/test02_rank.ipynb'
NOTEBOOK_SHA256 = 'dc7526bef020a0bec2ba23168c109a067c62013bf4a457a80514c3f7f9024551'
NOTEBOOK_SHA256_RULE = ('sha256 of json.dumps([the source of every cell], ensure_ascii=False) with this value replaced by '
                        '64 zeros (colab/build_test02_rank_nb.py; tools/t02_rank.py can recompute it)')

# 和官方 notebook 的全部差異（prereg01 的差異表，沿用的格子照它）
DEVIATIONS = [
    "[①] 官方 cell 4：改用 `c10data.load_rows(..., part=\"fit\")` 讀訓練資料（5,767 人、720,307 盤），`part=\"inner\"` 讀內部驗證（640 人、73,556 盤），各自轉成 DataFrame。`load_rows` 會 assert 人數、盤數，並確認沒有任何一盤出現在本機驗證檔。結果存 `leak_check.json`（每次執行各記一筆）。SMOKE 時等 `load_rows` 全部通過，才用固定 seed 縮小資料。",
    "[②] 官方 cell 4：取代 `df.iloc[:VAL_SIZE]` 的切法：`df_train`、`df_val` 各自 `sample(frac=1.0, random_state=42)` 打散之後，才寫分片。",
    "[③] 官方 cell 2、8、9：`VAL_SIZE` 改成內部驗證實際寫入的盤數：每個抽特徵的 worker 寫完後，把 `rows`、`written`、`bad` 寫進分片目錄的 `_count_w<N>.json`；`extract_split` 加總、核對後回傳；cell 9 設定 `VAL_SIZE` 並印出。cell 2 先設成 `None`。",
    "[④] 官方 cell 20、21：`predict()` 改讀打包檔的 `local/rank_val_questions.csv`，拿掉 public／private 迴圈（內層程式因此少一層縮排，進度訊息不印 split 名稱）。每題另存 10 維平均機率到 `rank_val_probs.csv`（完整 float；沒有可解析棋譜的題 `n_parsed=0`、機率欄留空）。官方的 argmax 結果改存成 `pred_official_argmax.csv`（官方是 `./submission_rank.csv`）。`predict()` 因此多了題目檔、機率檔路徑和 `limit` 三個參數；SMOKE 時只推論前 12 題。",
    "[營運性] 新增：最前面兩格 markdown：這份 notebook 的用途與操作說明，以及這張差異表。",
    "[營運性] 新增：新增的程式格：設定（Colab 或本機、Drive 路徑、SMOKE 縮小設定、結果紀錄工具）、準備（解壓打包檔並逐檔核對 SHA-256、確認沒有答案檔、解開訓練資料、把打包檔的 `tools/`、`tutorial/` 加進 import 路徑）、環境紀錄（`env.json`）、分片備份工具、設定紀錄與接續工具、結尾核對，以及把結果目錄裡權重以外的檔案打成 `rank_results.zip`（放在結果目錄旁，方便下載）。",
    "[營運性] 新增（準備格）：缺 `sgfmill` 或 `webdataset` 時才用 pip 安裝，版本固定為 `sgfmill==1.1.1`、`webdataset==1.0.2`（本機冒煙測試用的版本）。官方 notebook 只列出需要的套件，不安裝也不固定版本。Colab 內建的 torch、pandas、numpy、matplotlib 不動，實際版本記在 `env.json`。",
    "[營運性] 新增（設定格）：在 Linux 上明確指定 multiprocessing 用 `fork`。官方的 worker 函式定義在 notebook 內，依賴 fork；Python 3.14 起 Linux 預設改成 forkserver。3.13 以前本來就是 fork，行為不變。實際的啟動方式記在 `config.json`。",
    "[營運性] metadata：notebook metadata 加上 Colab 設定（GPU、A100、高 RAM），其餘沿用官方。官方格沿用官方的 cell id，新增的格用 `c10-rank-NN`；每格的 cell metadata 加 `c10` 對照欄（對應的官方格、是否改動）。官方原本就沒有存輸出。",
    "[營運性] 官方 cell markdown 0、3、5、19、22：官方的說明原文不動，在後面附加一段「**本版**」說明：路徑由設定格決定（0）、① ② 的讀法（3）、分片會備份到 Drive（5）、④ 只預測本機驗證題（19）、分數不在這裡算（22）。",
    "[營運性] 全部：工作目錄切到 `WORK`（Colab 是 `/content/c10`），官方的相對路徑 `./training`、`./rank-features` 照用。webdataset 會把 Windows 路徑的 `C:` 當成網址格式，本機冒煙測試只能用相對路徑。",
    "[營運性] 全部：改過的格在開頭或行內加了說明註解（標 ①–④ 或「本版」）。",
    "[營運性] 全部：紀錄檔都是每次執行各記一筆、不覆蓋前幾次：`timings.json`（各階段秒數）、`run_log.json`（每次執行與接續）、`env.json`、`leak_check.json`、`config.json`（頂層是固定說明與跨執行的累計狀態，每次執行的實際設定在 `runs`）。JSON 和 CSV 的 float 都寫完整精度。原生 Windows 的正式執行會直接報錯，因為官方的多進程依賴 Linux 的 fork。",
    "[營運性] 官方 cell 2：`CKPT_DIR` 改到 Drive 的結果目錄 `ckpt/`，拿掉 `TEST_DIR`。`NPROC` 從寫死的 8 改成 `os.cpu_count()`（官方註解寫「設成 CPU 核數」）。加上 SMOKE 縮小設定（只在 SMOKE 生效）。最後的 `print` 多印 `NPROC`、batch、epoch 數。",
    "[營運性] 官方 cell 4：不印官方的「全量 N 局」和全量的等級分布；改成逐一印 `fit`、`inner` 的人數與盤數，等級分布只印訓練集（`df_train`）。顯示前 3 列時不印棋譜欄，不讓棋譜內容進 notebook 輸出。",
    "[營運性] 官方 cell 6：示範用的第一盤改取 `df_train`（本版沒有合併的 `df`）。",
    "[營運性] 官方 cell 9：分片特徵抽完後備份到 Drive 的 `features_rank/`，全部核對後才寫完成標記。重跑時有完整且設定相符的備份，就複製回來、核對 SHA-256，不重抽；備份缺檔、大小或 SHA-256 不符時改成重抽。重抽前先清掉不完整的舊分片。",
    "[營運性] 官方 cell 11、12：`pick_workers` 另外記下讀出的筆數（寫進 `config.json`）。本機冒煙測試的 worker 候選數是 0（Windows 的 spawn 無法 pickle notebook 內定義的函式）。多 worker 對不上時照官方退回 0，並用 0 個 worker 再數一次；還是對不上 `VAL_SIZE` 就停下來，因為這表示 ③ 沒有成立。",
    "[營運性] 官方 cell 16：每個 epoch 結束，把 model、optimizer、epoch、最佳分數、history 存成 `ckpt/state.pt`。重跑時從最後完成的 epoch 接續；`state.pt` 讀不進來、或設定和上次不同，就停下來並說明處理方式。另外記最佳 epoch 的編號，結束時一併印出。history 多記 loss、步數、樣本數、秒數和第幾次執行。官方的 `epoch_N.pth`、`best_model.pth`、`history.csv` 照樣存在 `ckpt/`。",
    "[營運性] 官方 cell 16：`sync_best()` 把最佳 epoch 的 `epoch_N.pth` 複製成 `best_model.pth`：接續時做一次（中斷時可能留下寫一半的 `best_model.pth`），正常訓練結束時也做一次。複製來源和官方存的 `best_model.pth` 是同一個 epoch 結束時的同一份權重。找不到那個 `epoch_N.pth` 就停下來並說明處理方式。",
    "[營運性] 官方 cell 16：每次執行記下有沒有讀 `state.pt`、是否接續訓練、實際訓練了哪些 epoch（`run_log.json`、`config.json` 的 `runs`），頂層的 `resumed` 表示是否曾經接續訓練。",
    "[營運性] 官方 cell 18：訓練曲線另存 `ckpt/history.png`。非 Colab 時用不開視窗的 Agg 後端，不 `show()`，畫完就 `close()`。",
    "[營運性] 官方 cell 20：`predict()` 讀題目檔時 `question_id` 以字串讀入（`dtype={'question_id': str}`），代號原樣照抄，不會被轉成數字。每題 assert 非空的 `sgf_` 欄數等於 `num_games`，題目檔格式不對就停。兩者都不改變預測。",
    "[營運性] 官方 cell 21：推論用的權重、題目檔、題數寫進 `config.json`，秒數寫進 `timings.json`；輸出路徑指向 Drive 的結果目錄。"
]


def write_json(path, obj):
    # 先寫暫存檔再改名，中斷時不會留下寫一半的 JSON；float 用 repr，不四捨五入
    guard_write(path)   # 本版：寫進 prereg01 目錄的一律擋下
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=str)
    os.replace(tmp, path)


def read_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def read_runs(path, **init):
    # 每次執行各記一筆的紀錄檔（{'runs': [...], ...}），不覆蓋前幾次；舊格式（整份只有一次執行）當成一筆保留
    obj = read_json(path)
    if obj is None:
        return {'runs': [], **init}
    if not isinstance(obj, dict) or not isinstance(obj.get('runs'), list):
        return {'runs': [{'legacy': obj}], **init}
    return obj


# 每次「全部執行」算一次 run；接續次數和時間記在 run_log.json
RUN_LOG_PATH = os.path.join(RESULT_DIR, 'run_log.json')
run_log = read_runs(RUN_LOG_PATH, resume_count=0)
RUN = {'run': len(run_log['runs']) + 1, 'start': now_iso(), 'end': None, 'smoke': SMOKE,
       'in_colab': IN_COLAB, 'features': None, 'start_epoch': None, 'loaded_state': False,
       'resumed_training': False, 'epochs_trained': []}
run_log['runs'].append(RUN)
run_log['status'] = None   # 本版：這次執行結束時才寫（completed／seam_failed／readonly_changed）


def save_run_log():
    write_json(RUN_LOG_PATH, run_log)


TIMINGS_PATH = os.path.join(RESULT_DIR, 'timings.json')
timings = read_runs(TIMINGS_PATH)
TIMING = {'run': RUN['run'], 'start': RUN['start'], 'seconds': {}}
timings['runs'].append(TIMING)


def mark_time(stage, seconds):
    TIMING['seconds'][stage] = seconds
    write_json(TIMINGS_PATH, timings)


# config.json：頂層放固定的說明和跨執行的累計狀態；每次執行的實際設定各記一筆在 runs，不覆蓋前幾次
CONFIG_PATH = os.path.join(RESULT_DIR, 'config.json')
_old_config = read_runs(CONFIG_PATH)
CONFIG = {k: v for k, v in _old_config.items() if k != 'runs'}
CONFIG.update({
    'layout': '頂層：固定說明與跨執行的累計狀態；runs：每次執行各一筆的實際設定',
    'smoke': SMOKE, 'smoke_cfg': SMOKE_CFG if SMOKE else None,
    'prereg': 'docs/PREREG_02_OPENING_FUSION.md 第 9 節（rev 5，測試日；競賽二）',
    'rehearsal': REHEARSAL, 'test_kind': TEST_KIND,
    'notebook': NOTEBOOK_REL, 'notebook_sha256': NOTEBOOK_SHA256, 'notebook_sha256_rule': NOTEBOOK_SHA256_RULE,
    'base_notebook': {'path': 'colab/prereg02_rank.ipynb', 'sha256': '9fb1f9d06ba5ee5fbebfb6dfe022246888c93e7762e9a89e1adc99c9f1c12b0c'},
    'changes_from_prereg02': T02_CHANGES,
    'readonly_roots': READONLY_ROOTS,
    'official_notebook': 'third_party/aicup2026_tutorial/rank-prediction.ipynb',
    'deviations_from_official': DEVIATIONS,
    # resumed：是否曾有任何一次執行從 ckpt/state.pt 接續訓練。和 run_log.json 的 resume_count 同一個來源；
    # 已經訓練完、只重跑推論的那次不算
    'resumed': run_log['resume_count'] > 0, 'resume_count': run_log['resume_count'],
})
CONFIG['runs'] = _old_config['runs']
CONFIG_RUN = {'run': RUN['run'], 'start': RUN['start'], 'end': None, 'in_colab': IN_COLAB,
              'mp_start_method': MP_START_METHOD}
CONFIG['runs'].append(CONFIG_RUN)


def save_config(**kw):
    # 這次執行的實際設定：只更新 runs 裡這次的那一筆
    CONFIG_RUN.update(kw)
    write_json(CONFIG_PATH, CONFIG)


def save_config_summary(**kw):
    # 跨執行的累計狀態（是否曾接續、已完成幾個 epoch、最佳 epoch）：放頂層
    CONFIG.update(kw)
    write_json(CONFIG_PATH, CONFIG)


save_run_log()
save_config()
print('IN_COLAB', IN_COLAB, '| SMOKE', SMOKE, '| run', RUN['run'], '| multiprocessing', MP_START_METHOD)
print('WORK', WORK)
print('結果目錄', RESULT_DIR)
print('只讀：prereg01', P01_ROOT, '| prereg02', P02_ROOT)
print('測試集 zip：%s（種類 %s）' % (TEST_ZIP, TEST_KIND))

In [ ]:
# 準備：缺套件才安裝、把打包檔複製到 WORK 解壓並逐檔核對 SHA-256（本版不解開訓練資料）
import hashlib
import importlib.util
import subprocess
import tarfile
import zipfile

t0 = time.time()
if IN_COLAB:
    # 版本固定成本機冒煙測試用的版本
    need = [pkg for mod, pkg in (('sgfmill', 'sgfmill==1.1.1'), ('webdataset', 'webdataset==1.0.2'))
            if importlib.util.find_spec(mod) is None]
    if need:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *need])

BUNDLE_NAME = 'C10_bundle_v1'
BUNDLE_ZIP_SRC = os.path.join(DRIVE_DIR, BUNDLE_NAME + '.zip')
BUNDLE_ZIP = os.path.join(WORK, BUNDLE_NAME + '.zip')
BUNDLE = os.path.join(WORK, BUNDLE_NAME)


def sha256_file(path):
    d = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            d.update(chunk)
    return d.hexdigest()


def bundle_mismatches(manifest):
    return [k for k, v in manifest['files'].items()
            if not os.path.exists(os.path.join(BUNDLE, k)) or sha256_file(os.path.join(BUNDLE, k)) != v]


with zipfile.ZipFile(BUNDLE_ZIP_SRC) as z:
    MANIFEST = json.loads(z.read(BUNDLE_NAME + '/bundle_manifest.json'))

if read_json(os.path.join(BUNDLE, 'bundle_manifest.json')) == MANIFEST and not bundle_mismatches(MANIFEST):
    print('打包檔已解壓，SHA-256 全部相符，跳過解壓')
else:
    shutil.rmtree(BUNDLE, ignore_errors=True)
    shutil.copyfile(BUNDLE_ZIP_SRC, BUNDLE_ZIP)
    with zipfile.ZipFile(BUNDLE_ZIP) as z:
        z.extractall(WORK)
    bad = bundle_mismatches(MANIFEST)
    if bad:
        raise SystemExit('打包檔 SHA-256 不符：%s' % bad)
    print('打包檔解壓完成，%d 個檔案的 SHA-256 全部相符' % len(MANIFEST['files']))

# 紀律 7：Colab 上不能有答案檔
_all_files = [os.path.join(r, n) for r, _d, ns in os.walk(BUNDLE) for n in ns]
assert not any('answers' in p for p in _all_files), '打包檔裡有答案檔，停止'

# 本版（test02）：只推論，不解開訓練資料（底稿這裡解開 training/aicup2026_go_training.tar.xz 的整段拿掉）

# 打包檔裡的 c10data.py、官方 utils.py／network.py 加進 import 路徑
for _sub in ('tools', 'tutorial'):
    _p = os.path.join(BUNDLE, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)
LOCAL_DIR = os.path.join(BUNDLE, 'local')                      # 只有題目檔和候選庫
SPLIT_CSV = os.path.join(BUNDLE, 'split_v1', 'players_split.csv')
RANK_QUESTIONS = os.path.join(LOCAL_DIR, 'rank_val_questions.csv')
mark_time('prepare_bundle', time.time() - t0)
print('本機題目檔', sorted(os.listdir(LOCAL_DIR)))

In [ ]:
import glob
import os
import time
from multiprocessing import Pool, Process

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import webdataset as wds
from torch.utils.data import DataLoader

from utils import SGFParseRankPrediction
from network import GoRankResNet

print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '| webdataset', wds.__version__)

In [ ]:
# 環境紀錄：每次執行各記一筆到結果目錄的 env.json（接續時可能換到不同的機器）
import platform
from importlib import metadata


def _pkg_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def _ram_bytes():
    try:
        import psutil
        return psutil.virtual_memory().total
    except ImportError:
        pass
    if os.path.exists('/proc/meminfo'):
        with open('/proc/meminfo') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    return int(line.split()[1]) * 1024
    return None


_gpu = None
if torch.cuda.is_available():
    # 用 nvidia-smi 查，不在這裡初始化 CUDA（官方要到建模型時才初始化，之後才 fork 抽特徵的進程）
    try:
        _smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                               '--format=csv,noheader'], capture_output=True, text=True).stdout.strip().splitlines()
    except FileNotFoundError:
        _smi = None
    _gpu = {'nvidia_smi': _smi, 'device_count': torch.cuda.device_count()}
ENV_INFO = {
    'run': RUN['run'], 'start_time': RUN['start'], 'in_colab': IN_COLAB, 'smoke': SMOKE,
    'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
    'gpu': _gpu, 'cpu_count': os.cpu_count(), 'ram_bytes': _ram_bytes(),
    'multiprocessing_start_method': MP_START_METHOD,
    'packages': {n: _pkg_version(n) for n in ('pandas', 'numpy', 'webdataset', 'sgfmill', 'matplotlib')},
    'bundle_zip_bytes': os.path.getsize(BUNDLE_ZIP_SRC), 'bundle_manifest': MANIFEST,
}
ENV_PATH = os.path.join(RESULT_DIR, 'env.json')
_env_log = read_runs(ENV_PATH)
_env_log['runs'].append(ENV_INFO)
write_json(ENV_PATH, _env_log)
print(json.dumps({k: ENV_INFO[k] for k in ('torch', 'cuda', 'gpu', 'cpu_count', 'ram_bytes', 'packages')},
                 ensure_ascii=False))

In [ ]:
# 路徑：官方的相對路徑照用（工作目錄已切到 WORK）；checkpoint 改存到 Drive 的結果目錄，中斷後才能接續
TRAIN_DIR = './training'
FEAT_DIR = './rank-features'
# 本版（test02）：只推論，不存 checkpoint（拿掉 CKPT_DIR）

# 等級由弱到強。順序很重要：評分時的「相鄰一級」就是這個順序上的相鄰。
LEVELS = ['D', 'C', 'B', 'A', '1D', '2D', '3D', '4D', '5D', '6D']
RANK_TO_ID = {lv: i for i, lv in enumerate(LEVELS)}
ID_TO_RANK = {i: lv for lv, i in RANK_TO_ID.items()}

NPROC = os.cpu_count() # 特徵抽取的進程數，設成 CPU 核數
PER_SHARD = 1000       # 每個 webdataset 分片的樣本數
SEED = 42
VAL_SIZE = None        # ③ 第 1 節寫完分片後，改成內部驗證實際寫入的盤數

BATCH_SIZE = 256
LEARNING_RATE = 1e-3
NUM_EPOCHS = 5
NUM_CLASSES = len(LEVELS)

if SMOKE:
    # 只在本機冒煙測試生效：縮小資料量與步數，邏輯不變（縮小方式見第一個設定格的 SMOKE_CFG）
    NPROC = SMOKE_CFG['nproc']
    PER_SHARD = SMOKE_CFG['per_shard']
    BATCH_SIZE = SMOKE_CFG['batch_size']
    NUM_EPOCHS = SMOKE_CFG['num_epochs']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 本版（test02）：正式執行一定要有 GPU（接縫核對要和 prereg02 在 GPU 上算的逐盤輸出比）
if not SMOKE and DEVICE.type != 'cuda':
    raise SystemExit('沒有 GPU，停在這裡（還沒開始算）。處理方式：選單「執行階段 → 變更執行階段類型」選 G4 GPU'
                     '（灰色時選 A100 GPU，「大量 RAM」關掉）→ 儲存，再按一次「全部執行」。')
# 本版：GPU 型號只接受 G4（RTX PRO 6000）或 A100（PREREG_02 第 7 節第 8 條）。Colab 在選的 GPU 沒空時可能改派別的 GPU。
# 型號用環境紀錄格 nvidia-smi 查到的（不在這裡初始化 CUDA）
GPU_MODELS_OK = ('RTX PRO 6000', 'A100')
GPU_MODEL_NOW = gpu_name_of(ENV_INFO)
if not SMOKE and DEVICE.type == 'cuda' and not any(k in (GPU_MODEL_NOW or '') for k in GPU_MODELS_OK):
    raise SystemExit('這次拿到的 GPU 是 %s，不是 G4（RTX PRO 6000）或 A100，停在這裡（還沒開始算）。處理方式：'
                     '選單「執行階段 → 中斷連線並刪除執行階段」，再「執行階段 → 變更執行階段類型」選 G4 GPU'
                     '（灰色時選 A100 GPU，「大量 RAM」關掉）→ 儲存，再按一次「全部執行」。' % GPU_MODEL_NOW)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('device', DEVICE, '| GPU', GPU_MODEL_NOW, '| NPROC', NPROC)
RANK_TO_ID

In [ ]:
# 本版新增（preflight，在 GPU 關卡之後）：權重、接縫參照、測試集 zip、題目檔格式、舊輸出、Drive 空間、只讀目錄的指紋。
# 任何一項不符就用中文說明處理方式後停下，還沒開始算（但執行階段開著就會扣運算單元）。
import io

t0 = time.time()
DUMP_NAME, DUMP_SYMS = 'p02e9_d4', 8
SETS = ('seam_val', 'public', 'private')
# 產生器從 repo 讀出的釘選值（results/prereg02/rank/frozen_config.json、dumps/ 的完成標記與檔案）
W_PIN = '423581b4fddd5c2c160ebf691f4e86b9e00c4fe743d5cfb23df7891f4a2ca939'
W_LABEL_PIN = 'prereg02/rank/ckpt/epoch_9.pth'
SEAM_REF_SHA256_PIN = {"val_keys.npz": "055d29416bb79050128eea1e6944eb58376a751dc86780eee927bb94aca7ec8c", "val_p02e9_d4.npz": "ec0b729d805eb9f3458f7a497444b099b8ec47e5cec73703ae4b8bf9cf0a11b2"}
P02_REF_GPU_PIN = 'NVIDIA RTX PRO 6000 Blackwell Server Edition'
MIN_DRIVE_FREE_BYTES = 5 * 10 ** 8
N_TEST_QUESTIONS = 400
P02_RANK = os.path.join(P02_ROOT, 'rank')
P02_DUMPS = os.path.join(P02_RANK, 'dumps')
P02_HELP = ('處理方式：確認這個 Colab 用的是跑 prereg02 的同一個 Google 帳號，而且 Drive 上有 '
            'MyDrive/C10/results/prereg02/rank/ckpt/epoch_9.pth 和 rank/dumps/（val_keys.npz、val_p02e9_d4.npz 和它們的 _DONE_ 檔）。'
            '不要搬動、改名或修改 prereg02 的任何檔案。還是不對就把這段訊息截圖給 Claude。')

# ① 權重：prereg02 指定的 epoch_9.pth（SMOKE：prereg02_smoke 的完成標記記的那一份）
_m_ref = read_json(os.path.join(P02_DUMPS, '_DONE_val_%s.json' % DUMP_NAME))
if not _m_ref:
    raise SystemExit('找不到 prereg02 的 rank/dumps/_DONE_val_%s.json。%s' % (DUMP_NAME, P02_HELP))
if SMOKE:
    W_LABEL, WSHA = _m_ref['weights'], _m_ref['weights_sha256']
else:
    W_LABEL, WSHA = W_LABEL_PIN, W_PIN
    if (_m_ref.get('weights'), _m_ref.get('weights_sha256')) != (W_LABEL, WSHA):
        raise SystemExit('prereg02 的 rank/dumps/_DONE_val_%s.json 記的權重不是 %s。%s' % (DUMP_NAME, W_LABEL, P02_HELP))
if not W_LABEL.startswith('prereg02/'):
    raise SystemExit('prereg02 的完成標記記的權重路徑 %r 不在 prereg02 底下。請截圖給 Claude。' % W_LABEL)
W_PATH = os.path.join(P02_ROOT, *W_LABEL[len('prereg02/'):].split('/'))
if not os.path.isfile(W_PATH):
    raise SystemExit('找不到權重 %s。%s' % (W_PATH, P02_HELP))
if sha256_path(W_PATH) != WSHA:
    raise SystemExit('權重 %s 的 SHA-256 不是登記的 %s…。%s' % (W_PATH, WSHA[:12], P02_HELP))

# ② 接縫參照：prereg02 在 Drive 上的本機驗證題逐盤輸出（正式執行釘 SHA-256；SMOKE 核對完成標記）
SEAM_REF = {}
for _n in ('val_keys.npz', 'val_%s.npz' % DUMP_NAME):
    _p = os.path.join(P02_DUMPS, _n)
    if not os.path.isfile(_p):
        raise SystemExit('找不到接縫參照 %s。%s' % (_p, P02_HELP))
    SEAM_REF[_n] = sha256_path(_p)
    _want = ((read_json(os.path.join(P02_DUMPS, '_DONE_%s.json' % _n[:-4])) or {}).get('sha256') if SMOKE
             else SEAM_REF_SHA256_PIN[_n])
    if SEAM_REF[_n] != _want:
        raise SystemExit('接縫參照 %s 的 SHA-256 不符。%s' % (_p, P02_HELP))
with np.load(os.path.join(P02_DUMPS, 'val_%s.npz' % DUMP_NAME), allow_pickle=False) as _z:
    if str(_z['weights_sha256']) != WSHA:
        raise SystemExit('接縫參照 val_%s.npz 記的權重不是 %s…。%s' % (DUMP_NAME, WSHA[:12], P02_HELP))
P02_REF_GPU = _m_ref.get('gpu_name')   # 接縫核對的 GPU 規則：和算這次 seam_val dump 的 GPU 比
if not SMOKE and P02_REF_GPU != P02_REF_GPU_PIN:
    raise SystemExit('prereg02 的完成標記記的 GPU 是 %r，不是登記時的 %s。%s' % (P02_REF_GPU, P02_REF_GPU_PIN, P02_HELP))

# ③ 測試集 zip：只收 tests/ 底下的 6 個官方檔名和 MANIFEST.json，解到 WORK/tests；每個檔的 SHA-256、大小要等於 MANIFEST
TESTS_DIR = os.path.join(WORK, 'tests')
_allowed = {'tests/' + n for n in OFFICIAL_TEST_NAMES} | {TEST_MANIFEST_MEMBER}
shutil.rmtree(TESTS_DIR, ignore_errors=True)
os.makedirs(TESTS_DIR)
with zipfile.ZipFile(TEST_ZIP) as _z:
    _members = [i.filename for i in _z.infolist() if not i.filename.endswith('/')]
    _bad = [m for m in _members if m not in _allowed]
    _absent = sorted(_allowed - set(_members))
    if _bad or _absent or len(_members) != len(set(_members)):
        raise SystemExit('測試集 zip 的內容不對（多出：%s；缺少：%s）。%s' % (_bad[:5], _absent, TEST_HELP))
    for _m in _members:
        with open(os.path.join(TESTS_DIR, _m[len('tests/'):]), 'wb') as _f:
            _f.write(_z.read(_m))
TEST_ZIP_SHA = sha256_path(TEST_ZIP)
TEST_MANIFEST_SHA = sha256_path(os.path.join(TESTS_DIR, 'MANIFEST.json'))
if read_json(os.path.join(TESTS_DIR, 'MANIFEST.json')) != TEST_MANIFEST:
    raise SystemExit('測試集 zip 的 MANIFEST 和第一格讀到的不同。%s' % TEST_HELP)
_mf = TEST_MANIFEST.get('files') or {}
_problems = []
if sorted(_mf) != sorted(OFFICIAL_TEST_NAMES):
    _problems.append('MANIFEST 列的檔名不是 6 個官方檔名：%s' % sorted(_mf))
for _n in OFFICIAL_TEST_NAMES:
    _p = os.path.join(TESTS_DIR, _n)
    _e = _mf.get(_n) or {}
    if sha256_path(_p) != _e.get('sha256') or os.path.getsize(_p) != _e.get('bytes'):
        _problems.append('%s 的 SHA-256 或大小和 MANIFEST 不同' % _n)
if _problems:
    raise SystemExit('測試集 zip 核對沒過：%s。%s' % ('；'.join(_problems), TEST_HELP))


def quick_check_questions(path, prefix):
    # T1 的一部分（競賽二的題目檔）：沒有 BOM、400 題、question_id 不重複且前綴對、num_games 是 5–20 的整數、
    # sgf_1..20／color_1..20 欄都在、剛好前 num_games 格有棋、這幾格的執色是 B/W。回傳 (問題清單, question_id 清單)
    problems = []
    with open(path, 'rb') as f:
        raw = f.read()
    if raw.startswith(b'\xef\xbb\xbf'):
        problems.append('檔頭有 BOM')
    csv.field_size_limit(10 ** 8)
    rd = csv.DictReader(io.StringIO(raw.decode('utf-8-sig'), newline=''))
    need = ['question_id', 'num_games'] + ['sgf_%d' % i for i in range(1, 21)] + ['color_%d' % i for i in range(1, 21)]
    missing = [c for c in need if c not in (rd.fieldnames or [])]
    if missing:
        return problems + ['缺欄位 %s' % missing[:5]], []
    ids = []
    for r in rd:
        q = r['question_id']
        ids.append(q)
        if not q.startswith(prefix):
            problems.append('%s 的前綴不是 %s' % (q, prefix))
        g = (r['num_games'] or '').strip()
        if not g.isdigit() or not 5 <= int(g) <= 20:
            problems.append('%s 的 num_games 是 %r' % (q, r['num_games']))
            continue
        n = int(g)
        filled = [i for i in range(1, 21) if (r['sgf_%d' % i] or '').strip()]
        if filled != list(range(1, n + 1)):
            problems.append('%s：有棋的格是 %s，num_games 是 %d' % (q, filled[:3], n))
        bad = [i for i in range(1, n + 1) if r['color_%d' % i] not in ('B', 'W')]
        if bad:
            problems.append('%s：第 %s 格的執色不是 B/W' % (q, bad[:3]))
    if len(ids) != N_TEST_QUESTIONS:
        problems.append('%d 題，應該是 %d 題' % (len(ids), N_TEST_QUESTIONS))
    if len(set(ids)) != len(ids):
        problems.append('question_id 有重複')
    return problems, ids


_ids = {}
for _set, _prefix in (('public', 'pub_'), ('private', 'priv_')):
    _pr, _ids[_set] = quick_check_questions(os.path.join(TESTS_DIR, TEST_FILES[_set]), _prefix)
    _problems += ['%s：%s' % (TEST_FILES[_set], x) for x in _pr[:10]]
if set(_ids['public']) & set(_ids['private']):
    _problems.append('public 和 private 有相同的 question_id')
if _problems:
    raise SystemExit('測試題目檔的格式檢查沒過（%d 項）：%s。處理方式：不要改檔，把這段訊息截圖給 Claude。'
                     % (len(_problems), '；'.join(_problems[:10])))

# ④-0 上一次執行的接縫核對沒過就停：已完成的逐盤輸出會被 dump_is_done 沿用，登記第 9 節第 2 條的
#     「換一個執行階段重跑一次」就變成拿同一份輸出再比一次。要重跑必須重算，所以先把整個結果資料夾改名保留。
_prev_seam = read_json(os.path.join(RESULT_DIR, 'seam_check.json'))
if _prev_seam is not None and _prev_seam.get('passed') is not True:
    raise SystemExit('上一次執行的接縫核對沒過（%s 記 passed = %r）。這個資料夾裡的逐盤輸出不能沿用，重跑必須重算。'
                     '處理方式：先把畫面截圖給 Claude；Claude 說要重跑時，在 Google 雲端硬碟把 %s 整個資料夾'
                     '改名成 rank_failed_1（同一層，數字照 Claude 給的），再按一次「全部執行」。'
                     % (os.path.join(RESULT_DIR, 'seam_check.json'), _prev_seam.get('passed'),
                        os.path.relpath(RESULT_DIR, os.path.dirname(DRIVE_DIR)).replace(os.sep, '/')))

# ④ 各組的題目檔與 _INPUT.json：已存在但不同就停（dump_is_done 只看列數，換過的題目檔會被當成已完成）
SET_QUESTIONS = {'seam_val': RANK_QUESTIONS, 'public': os.path.join(TESTS_DIR, TEST_FILES['public']),
                 'private': os.path.join(TESTS_DIR, TEST_FILES['private'])}
SET_INPUT, SET_ROWS = {}, {}
for _set in SETS:
    _q = SET_QUESTIONS[_set]
    with open(_q, newline='', encoding='utf-8-sig') as _f:
        _n_rows = sum(1 for _ in csv.DictReader(_f))
    SET_ROWS[_set] = min(_n_rows, SMOKE_CFG['questions']) if SMOKE else _n_rows
    _rec = {'set': _set, 'question_file': ('bundle local/' if _set == 'seam_val' else 'tests/') + os.path.basename(_q),
            'sha256': sha256_path(_q)}
    _dir = os.path.join(RESULT_DIR, _set, 'dumps')
    _ip = os.path.join(_dir, '_INPUT.json')
    _old = read_json(_ip)
    if _old is not None and _old != _rec:
        raise SystemExit('%s 已有逐盤輸出，但當時的題目檔和這次不同（%s）。不能沿用。處理方式：不要刪檔，把這段訊息截圖給 Claude。'
                         % (_set, _ip))
    if _old is None and os.path.isdir(_dir) and os.listdir(_dir):
        raise SystemExit('%s 有檔案但沒有 _INPUT.json，不知道是用哪個題目檔算的。處理方式：不要刪檔，把這段訊息截圖給 Claude。' % _dir)
    guard_write(_dir)
    os.makedirs(_dir, exist_ok=True)
    if _old is None:
        write_json(_ip, _rec)
    SET_INPUT[_set] = _rec

# ⑤ Drive 剩餘空間
try:
    DRIVE_FREE_BYTES = shutil.disk_usage(DRIVE_DIR).free
except OSError as e:
    DRIVE_FREE_BYTES = None
    print('注意：查不到 Drive 剩餘空間（%s: %s），請自己確認還有 0.5 GB 以上' % (type(e).__name__, e))
if DRIVE_FREE_BYTES is not None and DRIVE_FREE_BYTES < MIN_DRIVE_FREE_BYTES:
    raise SystemExit('Drive 只剩 %.2f GB，這份 notebook 要 0.5 GB 以上。處理方式：清出空間（Drive 的垃圾桶也要清空）後再按一次'
                     '「全部執行」。' % (DRIVE_FREE_BYTES / 1e9))

# ⑥ 只讀目錄的指紋（開頭）。重跑時必須和第一次執行記下的相同：不同代表 prereg01／prereg02 在兩次執行之間被改過
READONLY_BEFORE = {_name: readonly_fingerprint(_root) for _name, _root in READONLY_ROOTS.items()}
_before_path = os.path.join(RESULT_DIR, 'readonly_before.json')
_before_log = read_runs(_before_path)
if _before_log['runs'] and _before_log['runs'][0]['files'] != READONLY_BEFORE:
    _first = _before_log['runs'][0]['files']
    _changed = {_k: sorted(f for f in set(_first.get(_k, {})) | set(READONLY_BEFORE[_k])
                           if _first.get(_k, {}).get(f) != READONLY_BEFORE[_k].get(f)) for _k in READONLY_BEFORE}
    raise SystemExit('只讀目錄和第一次執行時不同（%s）。處理方式：不要再執行，把這段訊息截圖給 Claude。'
                     % {k: v[:5] for k, v in _changed.items() if v})
_before_log['runs'].append({'run': RUN['run'], 'time': now_iso(), 'roots': READONLY_ROOTS,
                            'n_files': {k: len(v) for k, v in READONLY_BEFORE.items()}, 'files': READONLY_BEFORE})
write_json(_before_path, _before_log)

save_config(preflight={'weights': W_LABEL, 'weights_sha256': WSHA, 'seam_reference_sha256': SEAM_REF,
                       'p02_reference_gpu': P02_REF_GPU, 'test_zip_sha256': TEST_ZIP_SHA,
                       'test_manifest_sha256': TEST_MANIFEST_SHA, 'test_kind': TEST_KIND, 'sets': SET_INPUT,
                       'drive_free_bytes': DRIVE_FREE_BYTES,
                       'readonly_files_fingerprinted': {k: len(v) for k, v in READONLY_BEFORE.items()}})
save_config_summary(gpu=GPU_MODEL_NOW, torch=torch.__version__, cudnn=ENV_INFO.get('cudnn'),
                    weights={'path': W_LABEL, 'sha256': WSHA},
                    test_bundle={'zip_sha256': TEST_ZIP_SHA, 'manifest_sha256': TEST_MANIFEST_SHA,
                                 'kind': TEST_KIND, 't1': TEST_MANIFEST.get('t1'),
                                 'files': {n: _mf[n]['sha256'] for n in OFFICIAL_TEST_NAMES}},
                    sets={s: {'questions_sha256': SET_INPUT[s]['sha256'], 'rows': SET_ROWS[s]} for s in SETS},
                    seam_reference={'gpu_name': P02_REF_GPU, 'sha256': SEAM_REF})
mark_time('preflight', time.time() - t0)
print('preflight 通過：權重 %s（%s…），測試集 zip %s（%s…），題目 %s，prereg02 的 GPU：%s'
      % (W_LABEL, WSHA[:12], TEST_KIND, TEST_ZIP_SHA[:12], SET_ROWS, P02_REF_GPU))

In [ ]:
# 本版新增：逐盤輸出（dump）工具（test02：沿用 prereg02 的工具，拿掉 inner 的部分）
# - 題目檔（本機驗證題或官方測試題）：照題目檔的列順序和 slot 1..num_games 解析，失敗的保留位置（logits 填 NaN）；
#   每題一個 batch，和 prereg01、prereg02 的推論相同，接縫核對才對得上
# - 每個檔先寫 .tmp 再改名，附 _DONE_<檔名>.json；重跑時完成標記、SHA-256、權重 SHA-256 都對得上就跳過
# - DUMP_DIR 由各組的格子在呼叫前指定（<組>/dumps）；底稿的函式把它當全域變數讀，本身一字不改
import io
import math
import random
import zipfile

DUMP_DIR = None                        # 本版：各組的格子在呼叫前指定（<組>/dumps）
DUMP_MAX_BYTES = 100 * 1024 * 1024   # 結果 zip 會跳過 ≥ 100 MB 的檔，所以每個 dump 都必須更小
GPU_BATCH = 64                         # 和 prereg01 推論相同
NUM_SYMS = 8


def d4(x, s):
    # 8 種對稱：s=0 原方向；s>=4 先在 dim 3 鏡射，再在 dims (2, 3) 旋轉 s%4 次（x 是 [batch, 500, 19, 19]）
    if s == 0:
        return x
    y = torch.flip(x, dims=(3,)) if s >= 4 else x
    return torch.rot90(y, k=s % 4, dims=(2, 3)).contiguous()


# 自我檢查：8 種兩兩不同，而且任兩種接連做仍在這 8 種之內（封閉，是一個群）
_t = torch.arange(2 * 19 * 19, dtype=torch.float32).reshape(1, 2, 19, 19)
_imgs = [d4(_t, s) for s in range(NUM_SYMS)]
assert all(not torch.equal(_imgs[a], _imgs[b]) for a in range(NUM_SYMS) for b in range(a))
assert all(any(torch.equal(d4(_imgs[a], b), c) for c in _imgs) for a in range(NUM_SYMS) for b in range(NUM_SYMS))


def load_net(path, expect_sha=None):
    # 整個檔讀進記憶體、算 SHA-256，再從同一份位元組載入：記下的 SHA 就是實際用的權重
    with open(path, 'rb') as f:
        raw = f.read()
    got = hashlib.sha256(raw).hexdigest()
    if expect_sha is not None and got != expect_sha:
        raise SystemExit('%s 的 SHA-256 和開頭記下的不同（執行中被改動？）。請把這段訊息截圖給 Claude。' % path)
    net = GoRankResNet(num_classes=NUM_CLASSES)
    net.load_state_dict(torch.load(io.BytesIO(raw), map_location='cpu'))
    net.eval().to(DEVICE)
    return net, got


def _dump_path(stem):
    return os.path.join(DUMP_DIR, stem + '.npz')


def _marker_path(stem):
    return os.path.join(DUMP_DIR, '_DONE_%s.json' % stem)


def dump_is_done(stem, rows, syms, weights_sha):
    m = read_json(_marker_path(stem))
    if not m or not os.path.exists(_dump_path(stem)):
        return False
    if (m.get('rows'), m.get('syms'), m.get('weights_sha256')) != (rows, syms, weights_sha):
        return False
    return sha256_path(_dump_path(stem)) == m.get('sha256')


def write_dump(stem, rows, syms, weights, weights_sha, seconds, arrays):
    # arrays：要存進 npz 的 {名稱: 陣列}；rows、syms、weights、weights_sha 記進完成標記
    # 舊的完成標記先拿掉 → 寫 .tmp → 檢查大小 → 改名 → 最後才寫完成標記
    path, marker = _dump_path(stem), _marker_path(stem)
    tmp = path + '.tmp'
    for p in (path, tmp, marker):
        guard_write(p)
    if os.path.exists(marker):
        os.remove(marker)
    with open(tmp, 'wb') as f:
        np.savez(f, **arrays)
    size = os.path.getsize(tmp)
    if size >= DUMP_MAX_BYTES:
        raise SystemExit('%s 有 %.1f MB，超過 100 MB（結果 zip 會漏掉它）。請截圖給 Claude。' % (stem, size / 1e6))
    os.replace(tmp, path)
    # gpu_name／torch／cudnn：算出這個 dump 的那次執行（重跑時可能換了 GPU；重現核對 ② 的 GPU 規則看這裡）
    write_json(marker, {'file': 'dumps/%s.npz' % stem, 'rows': rows, 'syms': syms, 'bytes': size,
                        'sha256': sha256_path(path), 'weights': weights, 'weights_sha256': weights_sha,
                        'seconds': seconds, 'time': now_iso(), 'run': RUN['run'],
                        'gpu_name': gpu_name_of(ENV_INFO), 'torch': torch.__version__, 'cudnn': ENV_INFO.get('cudnn'),
                        'device': str(DEVICE)})


def require_gpu_for(stem):
    # 正式執行不在 CPU 上算逐盤輸出（要好幾天）；收尾開關打開又沒有 GPU 時，缺的 dump 在這裡停下
    if not SMOKE and DEVICE.type != 'cuda':
        raise SystemExit('%s 還沒算好，但這個執行階段沒有 GPU，停在這裡。處理方式：選單「執行階段 → 變更執行階段類型」'
                         '選 G4 GPU（灰色時選 A100 GPU，「大量 RAM」關掉）→ 儲存，再按一次「全部執行」。' % stem)


def record_readonly_after():
    # 本版：兩個只讀目錄（prereg01、prereg02）的指紋（結尾，或接縫核對沒過時）：和這次開頭記的比，寫進 readonly_after.json；
    # 回傳 (指紋, {目錄: 不同的檔})
    after = {name: readonly_fingerprint(root) for name, root in READONLY_ROOTS.items()}
    path = os.path.join(RESULT_DIR, 'readonly_after.json')
    log = read_runs(path)
    changed = {name: sorted(k for k in set(READONLY_BEFORE[name]) | set(after[name])
                            if READONLY_BEFORE[name].get(k) != after[name].get(k)) for name in after}
    changed = {k: v for k, v in changed.items() if v}
    log['runs'].append({'run': RUN['run'], 'time': now_iso(), 'roots': READONLY_ROOTS, 'files': after,
                        'n_files': {k: len(v) for k, v in after.items()},
                        'same_as_before': not changed, 'changed_files': changed})
    write_json(path, log)
    return after, changed


def pack_results(zip_name):
    # 結果目錄裡除了權重和暫存檔以外的檔案，打成一個 zip 放在結果目錄旁邊（prereg02/<zip_name>）；回傳 (路徑, 跳過的大檔)
    path = guard_write(os.path.join(os.path.dirname(RESULT_DIR), zip_name))
    tmp = guard_write(path + '.tmp')
    skipped = []
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(RESULT_DIR):
            dirs.sort()
            for f in sorted(files):
                p = os.path.join(root, f)
                if f.endswith(('.pth', '.pt', '.tmp')):
                    continue   # 權重與暫存檔不帶
                if os.path.getsize(p) > 100 * 1024 * 1024:
                    skipped.append(os.path.relpath(p, RESULT_DIR))
                    continue
                zf.write(p, os.path.relpath(p, os.path.dirname(RESULT_DIR)))
    os.replace(tmp, path)
    if skipped:
        print('注意：這些檔超過 100 MB，沒有打包：%s' % skipped)
    return path, skipped


def flush_drive():
    # Colab：下載 zip、刪除執行階段之前，先把 Drive 的寫入全部送出再卸載（權重、state.pt、zip 才確定都在 Drive 上）
    if IN_COLAB:
        drive.flush_and_unmount()
        print('Drive 已寫入完成並卸載；要再跑請重新「全部執行」')


def write_keys(stem, arrays):
    # key 檔：已有完整的就讀回來比對，必須完全相同（不同代表列順序變了，已存的 dump 會對錯盤）
    rows = len(next(iter(arrays.values())))
    if dump_is_done(stem, rows, None, None):
        with np.load(_dump_path(stem), allow_pickle=False) as z:
            same = sorted(z.files) == sorted(arrays) and all(
                z[k].dtype == v.dtype and np.array_equal(z[k], v) for k, v in arrays.items())
        if not same:
            raise SystemExit('%s 和這次算出的 key 不同，已存的逐盤輸出可能對錯盤。處理方式：不要刪檔，把這段訊息截圖給 Claude。'
                             % stem)
        print('  %s：已存在且相同，跳過' % stem)
        return 'skipped'
    t0 = time.time()
    write_dump(stem, rows, None, None, None, time.time() - t0, arrays)
    print('  %s：寫出 %d 列' % (stem, rows))
    return 'written'


_infer_parser = SGFParseRankPrediction()


def sgf_to_features(sgf_content):
    # 給 multiprocessing.Pool 用的頂層函式（和 prereg01 推論相同）
    feat, err = _infer_parser.features_from_sgf_content(sgf_content)
    return None if err is not None else feat


QSETS = {}   # 本版：parse_questions() 的結果，依題目檔路徑快取（每個檔只解析一次；各組的格子用完就釋放）


def parse_questions(path):
    # 本版：題目檔（本機驗證題或官方測試題）：題目檔的列順序、slot 1..num_games；SMOKE 時和 prereg01 冒煙一樣只取前幾題
    if path in QSETS:
        return QSETS[path]
    t0 = time.time()
    test = pd.read_csv(path, dtype={'question_id': str})
    if SMOKE:
        test = test.head(SMOKE_CFG['questions'])
    sgf_cols = [c for c in test.columns if c.startswith('sgf_')]
    qids, slots, colors, sgfs, spans = [], [], [], [], []
    for _, r in test.iterrows():
        # 和 prereg01 一樣取非空的 sgf_ 欄；本版另外確認它們剛好是 sgf_1..sgf_num_games，slot 才對得回執色
        n = int(r['num_games'])
        cols = [c for c in sgf_cols if pd.notna(r[c]) and str(r[c]).strip()]
        assert cols == ['sgf_%d' % j for j in range(1, n + 1)], (r['question_id'], cols, n)
        spans.append((r['question_id'], len(sgfs), n))
        for j in range(1, n + 1):
            c = r['color_%d' % j]
            assert c in ('B', 'W'), (r['question_id'], j, c)
            qids.append(r['question_id'])
            slots.append(j)
            colors.append(c)
            sgfs.append(r['sgf_%d' % j])
    with Pool(NPROC) as pool:
        feats = pool.map(sgf_to_features, sgfs, chunksize=8)
    parsed = np.array([f is not None for f in feats], dtype=bool)
    QSETS[path] = {'spans': spans, 'feats': feats,
                   'keys': {'question_id': np.array(qids, dtype=str), 'slot': np.array(slots, dtype=np.int32),
                            'color': np.array(colors, dtype=str), 'parsed': parsed}}
    mark_time('parse_questions %s' % os.path.basename(path), time.time() - t0)
    print('題目檔 %s：%d 題、%d 盤，解析失敗 %d 盤，%.0f 秒'
          % (os.path.basename(path), len(spans), len(feats), int((~parsed).sum()), time.time() - t0), flush=True)
    return QSETS[path]


def dump_questions(net, path, name, syms, weights, weights_sha):
    # 本版：底稿的 dump_val 改成指定題目檔；檔名仍是 'val_' + 名稱（在 test02 裡只代表「一組題目的逐盤輸出」），寫到目前的 DUMP_DIR
    V = parse_questions(path)
    stem, n = 'val_' + name, len(V['feats'])
    if dump_is_done(stem, n, syms, weights_sha):
        print('  %s：已完成，跳過' % stem)
        return 'skipped'
    require_gpu_for(stem)
    t0 = time.time()
    logits = np.full((n, syms, NUM_CLASSES), np.nan, dtype=np.float32)
    with torch.no_grad():
        for _qid, start, k in V['spans']:
            ok = [start + j for j in range(k) if V['feats'][start + j] is not None]
            if not ok:
                continue
            # 和 prereg01 相同：一題的棋疊成一個 batch（每批最多 GPU_BATCH 盤）
            x = torch.from_numpy(np.stack([V['feats'][g] for g in ok])).float()
            for i in range(0, len(x), GPU_BATCH):
                xb = x[i:i + GPU_BATCH].to(DEVICE)
                rows = ok[i:i + GPU_BATCH]
                for s in range(syms):
                    logits[rows, s] = net(d4(xb, s)).float().cpu().numpy()
    write_dump(stem, n, syms, weights, weights_sha, time.time() - t0,
               {'logits': logits, 'weights_sha256': np.array(weights_sha), 'syms': np.array(syms)})
    print('  %s：%d 盤 × %d 種方向，%.0f 秒' % (stem, n, syms, time.time() - t0), flush=True)
    return 'written'


_E1 = math.exp(-1)


def _argmax_first(values):
    # 最大值的位置；同分取前面的（和 numpy.argmax、tools/decide.py 相同）
    best = 0
    for i in range(1, len(values)):
        if values[i] > values[best]:
            best = i
    return best


def rule_b(p):
    # 規則 B（和 tools/decide.py 的 rule_b 相同）：使 p_k + e^-1·(p_{k-1} + p_{k+1}) 最大的等級，兩端外側算 0
    n = len(p)
    return LEVELS[_argmax_first([p[k] + _E1 * ((p[k - 1] if k > 0 else 0.0) + (p[k + 1] if k < n - 1 else 0.0))
                                 for k in range(n)])]


NET = None      # 本版：三組共用的網路，第一組的格子載入一次
SET_LOG = {}    # 本版：每組的逐盤輸出紀錄（寫進 config.json）
print('dump 工具就緒：權重 %s（SHA-256 %s…），%d 種方向' % (W_LABEL, WSHA[:12], NUM_SYMS))

In [ ]:
SET = 'seam_val'   # 本版新增：這一格處理的題目組（seam_val、public、private 三格只有這一行不同）
# 一組題目的逐盤輸出（prereg02 的 epoch_9.pth，8 種對稱），寫到 <組>/dumps；已完成就跳過
t0 = time.time()
_path = SET_QUESTIONS[SET]
DUMP_DIR = guard_write(os.path.join(RESULT_DIR, SET, 'dumps'))   # 底稿的 dump 函式把 DUMP_DIR 當全域變數讀
os.makedirs(DUMP_DIR, exist_ok=True)
if read_json(os.path.join(DUMP_DIR, '_INPUT.json')) != SET_INPUT[SET]:
    raise SystemExit('%s 的 _INPUT.json 和 preflight 記的不同。請把這段訊息截圖給 Claude。' % SET)
if NET is None:
    NET, _ = load_net(W_PATH, expect_sha=WSHA)
_V = parse_questions(_path)
_st = [write_keys('val_keys', _V['keys']), dump_questions(NET, _path, DUMP_NAME, DUMP_SYMS, W_LABEL, WSHA)]
SET_LOG[SET] = {'questions': len(_V['spans']), 'games': len(_V['feats']),
                'games_unparsed': int((~_V['keys']['parsed']).sum()), 'keys': _st[0], 'dump': _st[1],
                'seconds': time.time() - t0}
del _V
QSETS.pop(_path, None)   # 解析結果用完就釋放記憶體
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
save_config(dumps=SET_LOG)
mark_time('dump_%s' % SET, time.time() - t0)
print('%s：%s' % (SET, SET_LOG[SET]))

In [ ]:
SET = 'public'   # 本版新增：這一格處理的題目組（seam_val、public、private 三格只有這一行不同）
# 一組題目的逐盤輸出（prereg02 的 epoch_9.pth，8 種對稱），寫到 <組>/dumps；已完成就跳過
t0 = time.time()
_path = SET_QUESTIONS[SET]
DUMP_DIR = guard_write(os.path.join(RESULT_DIR, SET, 'dumps'))   # 底稿的 dump 函式把 DUMP_DIR 當全域變數讀
os.makedirs(DUMP_DIR, exist_ok=True)
if read_json(os.path.join(DUMP_DIR, '_INPUT.json')) != SET_INPUT[SET]:
    raise SystemExit('%s 的 _INPUT.json 和 preflight 記的不同。請把這段訊息截圖給 Claude。' % SET)
if NET is None:
    NET, _ = load_net(W_PATH, expect_sha=WSHA)
_V = parse_questions(_path)
_st = [write_keys('val_keys', _V['keys']), dump_questions(NET, _path, DUMP_NAME, DUMP_SYMS, W_LABEL, WSHA)]
SET_LOG[SET] = {'questions': len(_V['spans']), 'games': len(_V['feats']),
                'games_unparsed': int((~_V['keys']['parsed']).sum()), 'keys': _st[0], 'dump': _st[1],
                'seconds': time.time() - t0}
del _V
QSETS.pop(_path, None)   # 解析結果用完就釋放記憶體
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
save_config(dumps=SET_LOG)
mark_time('dump_%s' % SET, time.time() - t0)
print('%s：%s' % (SET, SET_LOG[SET]))

In [ ]:
SET = 'private'   # 本版新增：這一格處理的題目組（seam_val、public、private 三格只有這一行不同）
# 一組題目的逐盤輸出（prereg02 的 epoch_9.pth，8 種對稱），寫到 <組>/dumps；已完成就跳過
t0 = time.time()
_path = SET_QUESTIONS[SET]
DUMP_DIR = guard_write(os.path.join(RESULT_DIR, SET, 'dumps'))   # 底稿的 dump 函式把 DUMP_DIR 當全域變數讀
os.makedirs(DUMP_DIR, exist_ok=True)
if read_json(os.path.join(DUMP_DIR, '_INPUT.json')) != SET_INPUT[SET]:
    raise SystemExit('%s 的 _INPUT.json 和 preflight 記的不同。請把這段訊息截圖給 Claude。' % SET)
if NET is None:
    NET, _ = load_net(W_PATH, expect_sha=WSHA)
_V = parse_questions(_path)
_st = [write_keys('val_keys', _V['keys']), dump_questions(NET, _path, DUMP_NAME, DUMP_SYMS, W_LABEL, WSHA)]
SET_LOG[SET] = {'questions': len(_V['spans']), 'games': len(_V['feats']),
                'games_unparsed': int((~_V['keys']['parsed']).sum()), 'keys': _st[0], 'dump': _st[1],
                'seconds': time.time() - t0}
del _V
QSETS.pop(_path, None)   # 解析結果用完就釋放記憶體
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
save_config(dumps=SET_LOG)
mark_time('dump_%s' % SET, time.time() - t0)
print('%s：%s' % (SET, SET_LOG[SET]))

In [ ]:
# 本版新增：接縫核對（PREREG_02 第 9 節第 2 條）：這次重算的 seam_val 逐盤輸出，和 Drive 上 prereg02 的比
# 通過條件：key 陣列完全相同；原方向逐盤 softmax 的每題 float32 平均（C0）最大差 ≤ tol（算這份 dump 的 GPU 型號和
# prereg02 相同時 1e-4，不同時 1e-3）；規則 B（C0）不同的題 ≤ 4（SMOKE 依題數等比例，至少 1）
t0 = time.time()
SEAM_KEYS = ('question_id', 'slot', 'color', 'parsed')


def _load_npz(path):
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}


def question_spans(qids):
    # key 陣列的題目區段 [(question_id, 開始列, 盤數)]；同一題的 slot 必須相鄰
    out, seen = [], set()
    for i, q in enumerate(qids.tolist()):
        if out and out[-1][0] == q:
            out[-1][2] += 1
            continue
        assert q not in seen, ('同一題的 slot 不相鄰', q)
        seen.add(q)
        out.append([q, i, 1])
    return out


def c0_of(L, spans, parsed):
    # C0：每盤原方向 softmax、同題在 float32 下平均（和 prereg01 推論、prereg02 的重現核對相同）；沒有可解析的盤是 None
    out = []
    for _q, start, k in spans:
        ok = [start + j for j in range(k) if parsed[start + j]]
        out.append([float(v) for v in torch.from_numpy(L[ok, 0, :]).softmax(1).numpy().mean(0)] if ok else None)
    return out


def seam_compare(new, ref, same_gpu, smoke):
    # new、ref：{'keys': {...}, 'logits': 陣列}。回傳紀錄；'passed' 是登記的判定
    keys_equal = {k: bool(new['keys'][k].shape == ref['keys'][k].shape and np.array_equal(new['keys'][k], ref['keys'][k]))
                  for k in SEAM_KEYS}
    L1, L0 = new['logits'], ref['logits']
    rec = {'keys_equal': keys_equal, 'same_gpu_as_prereg02': bool(same_gpu), 'tol': 1e-4 if same_gpu else 1e-3,
           'bitwise_equal': bool(all(keys_equal.values()) and L1.dtype == L0.dtype and L1.shape == L0.shape
                                 and L1.tobytes() == L0.tobytes()),
           'max_abs_logit_diff': None, 'max_abs_c0_diff': None, 'problems': []}
    if not all(keys_equal.values()) or L1.shape != L0.shape:
        rec['problems'].append('key 陣列或 logits 的形狀和 prereg02 不同：%s' % keys_equal)
        rec['passed'] = False
        return rec
    parsed = new['keys']['parsed'].astype(bool)
    rec['max_abs_logit_diff'] = (float(np.max(np.abs(L1[parsed].astype(np.float64) - L0[parsed].astype(np.float64))))
                                 if parsed.any() else 0.0)
    spans = question_spans(new['keys']['question_id'])
    c1, c0 = c0_of(L1, spans, parsed), c0_of(L0, spans, parsed)
    diffs = [max(abs(a - b) for a, b in zip(x, y)) for x, y in zip(c1, c0) if x is not None]
    rec['max_abs_c0_diff'] = max(diffs) if diffs else 0.0
    b_diff = [s[0] for s, x, y in zip(spans, c1, c0) if x is not None and rule_b(x) != rule_b(y)]
    b_allowed = 4 if not smoke else max(1, math.ceil(4 * len(spans) / 1589))
    rec.update(questions=len(spans), games=int(len(parsed)), games_parsed=int(parsed.sum()),
               rule_b_diff={'questions': len(b_diff), 'allowed': b_allowed, 'question_ids': b_diff[:50]})
    rec['passed'] = bool(rec['max_abs_c0_diff'] <= rec['tol'] and len(b_diff) <= b_allowed)
    return rec


_new_dir = os.path.join(RESULT_DIR, 'seam_val', 'dumps')
_new = {'keys': _load_npz(os.path.join(_new_dir, 'val_keys.npz')),
        'logits': _load_npz(os.path.join(_new_dir, 'val_%s.npz' % DUMP_NAME))['logits']}
_ref = {'keys': _load_npz(os.path.join(P02_DUMPS, 'val_keys.npz')),
        'logits': _load_npz(os.path.join(P02_DUMPS, 'val_%s.npz' % DUMP_NAME))['logits']}
_m_new = read_json(os.path.join(_new_dir, '_DONE_val_%s.json' % DUMP_NAME)) or {}
SEAM_DUMP_GPU = _m_new.get('gpu_name')   # 算這份 seam_val dump 的那次執行的 GPU（接續時可能不是這次）
_same_gpu = SEAM_DUMP_GPU == P02_REF_GPU and (SMOKE or SEAM_DUMP_GPU is not None)
SEAM = seam_compare(_new, _ref, _same_gpu, SMOKE)
del _new, _ref
# 三組的逐盤輸出必須是同一個執行環境算的：中斷後接續時，已完成的組會被 dump_is_done 沿用，public、private 可能是
# 別次執行（別的 GPU、torch、cuDNN）算的，接縫就沒涵蓋產生測試輸出的環境（登記第 9 節第 2 條）。不同就算接縫沒過
SEAM_ENV_KEYS = ('gpu_name', 'torch', 'cudnn')
SEAM_DUMP_ENV = {_s: {_k: (read_json(os.path.join(RESULT_DIR, _s, 'dumps', '_DONE_val_%s.json' % DUMP_NAME)) or {}).get(_k)
                      for _k in SEAM_ENV_KEYS} for _s in SETS}
_env_mixed = [_s for _s in SETS if SEAM_DUMP_ENV[_s] != SEAM_DUMP_ENV['seam_val']]
if _env_mixed:
    SEAM['problems'].append('%s 的逐盤輸出和 seam_val 的不是同一個執行環境算的（GPU 型號、torch、cuDNN：%s）'
                            % ('、'.join(_env_mixed), SEAM_DUMP_ENV))
    SEAM['passed'] = False
SEAM.update({
    'what': 'test02 notebook seam (PREREG_02 section 9 item 2): seam_val dump vs prereg02 val_p02e9_d4.npz on Drive',
    'run': RUN['run'], 'time': now_iso(), 'smoke': SMOKE,
    'tol_rule': '1e-4 when the GPU that wrote the seam_val dump (its _DONE_ marker) is the prereg02 dump GPU, else 1e-3',
    'gpu_this_run': GPU_MODEL_NOW, 'torch_this_run': torch.__version__, 'cudnn_this_run': ENV_INFO.get('cudnn'),
    'seam_dump': {k: _m_new.get(k) for k in ('gpu_name', 'torch', 'cudnn', 'run', 'sha256', 'weights_sha256')},
    'prereg02_dump': {k: _m_ref.get(k) for k in ('gpu_name', 'torch', 'cudnn', 'run', 'sha256', 'weights_sha256')},
    'reference_sha256': SEAM_REF,
    'dump_env': SEAM_DUMP_ENV, 'dump_env_same': not _env_mixed,
    'dump_env_rule': 'the val_p02e9_d4 dumps of seam_val, public and private must carry the same gpu_name, torch and '
                     'cudnn in their _DONE markers (a resumed run reuses finished sets)'})
_seam_path = os.path.join(RESULT_DIR, 'seam_check.json')
_prev = read_json(_seam_path)
SEAM['earlier_runs'] = ((_prev.get('earlier_runs') or []) + [{k: v for k, v in _prev.items() if k != 'earlier_runs'}]
                        if _prev else [])
write_json(_seam_path, SEAM)
save_config(seam_check={k: SEAM.get(k) for k in ('passed', 'tol', 'bitwise_equal', 'max_abs_c0_diff',
                                                 'max_abs_logit_diff', 'rule_b_diff', 'same_gpu_as_prereg02',
                                                 'dump_env_same')})
mark_time('seam_check', time.time() - t0)
print('接縫核對：C0 最大差 %s（容許 %g；算 dump 的 GPU %s，prereg02 %s），規則 B 不同 %s，逐位元組相同 %s → %s'
      % (SEAM['max_abs_c0_diff'], SEAM['tol'], SEAM_DUMP_GPU, P02_REF_GPU, SEAM.get('rule_b_diff'),
         SEAM['bitwise_equal'], '通過' if SEAM['passed'] else '沒過'))
if not SEAM['passed']:
    # 停下前照樣比對只讀目錄的指紋、打包（查原因要用），並把 Drive 的寫入送出
    _after, _ro_changed = record_readonly_after()
    RUN.update(end=now_iso(), status='seam_failed')
    run_log['status'] = 'seam_failed'
    save_config_summary(status='seam_failed')
    save_run_log()
    _zip, _ = pack_results('test02_rank_results.zip')
    flush_drive()
    raise SystemExit('接縫核對沒過%s：這次執行的測試輸出一律不用、不上傳（PREREG_02 第 9 節第 2 條）。處理方式：不要自己重跑；'
                     '下載 %s，連同這段訊息的截圖一起給 Claude。'
                     % (('（%s）' % '；'.join(SEAM['problems'])) if SEAM['problems'] else '', _zip))

In [ ]:
# 本版改寫的結尾：核對三組的逐盤輸出，比對兩個只讀目錄前後的指紋，補完 config／timings／run_log，
# 打包 test02_rank_results.zip（不含權重），把 Drive 的寫入送出
t0 = time.time()
DUMP_CHECK = {}
for SET in SETS:
    _dir = os.path.join(RESULT_DIR, SET, 'dumps')
    _chk = {}
    for _stem in ('val_keys', 'val_' + DUMP_NAME):
        _path, _m = os.path.join(_dir, _stem + '.npz'), read_json(os.path.join(_dir, '_DONE_%s.json' % _stem))
        _sha = sha256_path(_path) if os.path.exists(_path) else None
        assert _m is not None and _sha == _m['sha256'], ('完成標記不符', SET, _stem)
        assert os.path.getsize(_path) < DUMP_MAX_BYTES, (SET, _stem)
        _chk[_stem] = {'rows': _m['rows'], 'bytes': os.path.getsize(_path), 'sha256': _sha,
                       'gpu_name': _m.get('gpu_name'), 'torch': _m.get('torch'), 'cudnn': _m.get('cudnn'),
                       'run': _m.get('run')}
    with np.load(os.path.join(_dir, 'val_keys.npz'), allow_pickle=False) as _z:
        _keys = {k: _z[k] for k in _z.files}
    _rows = len(_keys['question_id'])
    assert sorted(_keys) == ['color', 'parsed', 'question_id', 'slot'] and all(len(v) == _rows for v in _keys.values()), SET
    _parsed = _keys['parsed'].astype(bool)
    with np.load(os.path.join(_dir, 'val_%s.npz' % DUMP_NAME), allow_pickle=False) as _z:
        _L = _z['logits']
        assert _L.dtype == np.float32 and _L.shape == (_rows, DUMP_SYMS, NUM_CLASSES), (SET, _L.shape)
        assert str(_z['weights_sha256']) == WSHA and int(_z['syms']) == DUMP_SYMS, SET
        assert np.array_equal(np.isnan(_L).all(axis=(1, 2)), ~_parsed) and not np.isnan(_L[_parsed]).any(), SET
    assert _rows == _chk['val_keys']['rows'] == _chk['val_' + DUMP_NAME]['rows'], SET
    assert read_json(os.path.join(_dir, '_INPUT.json')) == SET_INPUT[SET], SET
    _qids = _keys['question_id'].tolist()
    DUMP_CHECK[SET] = {'questions': len(set(_qids)), 'games': _rows, 'games_unparsed': int((~_parsed).sum()),
                       'files': _chk, 'input': SET_INPUT[SET]}
    assert DUMP_CHECK[SET]['questions'] == SET_ROWS[SET], (SET, DUMP_CHECK[SET]['questions'], SET_ROWS[SET])
    del _keys, _L

# 只讀目錄的指紋（結尾）：必須和這次開頭記的完全相同
READONLY_AFTER, _ro_changed = record_readonly_after()
STATUS = 'completed' if not _ro_changed else 'readonly_changed'
OUTPUTS = {
    'dumps': DUMP_CHECK, 'seam_check_passed': SEAM['passed'], 'seam_bitwise_equal': SEAM['bitwise_equal'],
    'readonly_unchanged': not _ro_changed, 'readonly_changed_files': _ro_changed,
    'files': {n: os.path.getsize(os.path.join(RESULT_DIR, n)) for n in sorted(os.listdir(RESULT_DIR))
              if os.path.isfile(os.path.join(RESULT_DIR, n))},
}
RUN.update(end=now_iso(), status=STATUS)
run_log['status'] = STATUS
save_config(outputs=OUTPUTS, end=RUN['end'])
save_config_summary(status=STATUS)
save_run_log()
mark_time('final_check', time.time() - t0)
mark_time('total_this_run', time.time() - RUN_START)
print(json.dumps({k: v for k, v in OUTPUTS.items() if k != 'dumps'}, ensure_ascii=False, indent=1))
print(json.dumps({s: {k: v for k, v in d.items() if k in ('questions', 'games', 'games_unparsed')}
                  for s, d in DUMP_CHECK.items()}, ensure_ascii=False))

# 結果打包：結果目錄裡除了權重和暫存檔以外的檔案，打成 test02/test02_rank_results.zip（zip 內路徑從 rank/ 起）
RESULTS_ZIP, _skipped_big = pack_results('test02_rank_results.zip')
with zipfile.ZipFile(RESULTS_ZIP) as _zf:
    _names = set(_zf.namelist())
_need = (['rank/' + n for n in ('config.json', 'run_log.json', 'env.json', 'timings.json', 'seam_check.json',
                                'readonly_before.json', 'readonly_after.json')]
         + ['rank/%s/dumps/%s' % (s, n) for s in SETS
            for n in ('val_keys.npz', 'val_%s.npz' % DUMP_NAME, '_DONE_val_keys.json', '_DONE_val_%s.json' % DUMP_NAME,
                      '_INPUT.json')])
_missing_in_zip = [n for n in _need if n not in _names]
assert not _missing_in_zip and not _skipped_big, ('沒進 zip', _missing_in_zip, _skipped_big)
_zip_mb = os.path.getsize(RESULTS_ZIP) / 1e6
flush_drive()
if _ro_changed:
    raise SystemExit('prereg01 或 prereg02 的目錄在這次執行中被改動（%s）。結果已打包在 %s，但先不要用；'
                     '把這段訊息截圖給 Claude。' % ({k: v[:5] for k, v in _ro_changed.items()}, RESULTS_ZIP))
print('完成。請下載這個檔回傳給 Claude（合併、校準、交件檔都在本機做）：%s（%.1f MB）' % (RESULTS_ZIP, _zip_mb))